In [73]:
from scipy import stats

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import plotly
import plotly.offline as py
from plotly.graph_objs import *
import plotly.graph_objs as go
plotly.offline.init_notebook_mode(connected=True)

In [74]:
# install package 
# !pip install --user plotly

In [75]:
# The code was removed by DSX for sharing.

In [76]:
# rename columns and load tb dataset

columns = ['country', 'year', 'e_pop_num', 'e_inc_num', 'e_inc_100k', 'e_inc_tbhiv_num', 'e_inc_tbhiv_100k', 'e_tbhiv_prct']
tb = pd.read_csv(get_object_storage_file_with_credentials_a4e60d264239489f8c0dbe5be7799f97('DefaultProjecteyeoffaustgmailcom', 'basic_TB_data.csv'))[columns]
tb.columns = ['country','year', 'population', 'tb_all_cases', 'tb_all_cases_100k', 'tb+hiv_cases', 'tb+hiv+cases_100k','%_tb+hiv_cases']

In [77]:
# merge tb and hiv dataset on country column
tb2015 = tb[tb['year'] == 2015].merge(hiv2015, on='country')
tb2010 = tb[tb['year'] == 2010].merge(hiv2010, on='country')
tb2005 = tb[tb['year'] == 2005].merge(hiv2005, on='country')
tb2000 = tb[tb['year'] == 2000].merge(hiv2000, on='country')

df_tb = pd.concat([tb2015, tb2010, tb2005, tb2000], axis=0)

# only concerned with rows that have data in hiv cases
df_tb = df_tb[df_tb['hiv_cases'] != 'No data']

In [78]:
# change hiv range brackets to number

def to_int(x):
    x = (x.split('[')[0][:-1]).replace(" ", "")
    if '&lt;' in x:
        x = x[4:]
    return int(x)
        

df_tb['hiv_cases'] = df_tb['hiv_cases'].apply(to_int)

# new column with hiv per 100k people
df_tb['hiv_cases_100k'] = np.round((df_tb['hiv_cases'] / df_tb['population']) * 100000)

# df_tb.head()

# Figure 1

In [79]:
###############################################
TITLE = 'Prevalence of HIV & TB<br>Source: WHO (2015)'
###############################################


only2015 = pd.read_csv(get_object_storage_file_with_credentials_a4e60d264239489f8c0dbe5be7799f97('DefaultProjecteyeoffaustgmailcom', 'basic_TB_data.csv'))[['country', 'iso3', 'year', 'e_inc_tbhiv_100k']]
only2015 = only2015[(only2015['year'] == 2015) & (only2015['e_inc_tbhiv_100k'].notnull())]


data = [ dict(
        type = 'choropleth',
        locations = only2015['iso3'],
        z = only2015['e_inc_tbhiv_100k'],
        text = only2015['country'],
        colorscale = [[0,"rgb(5, 10, 172)"],[0.35,"rgb(40, 60, 190)"],[0.5,"rgb(70, 100, 245)"],\
            [0.6,"rgb(90, 120, 245)"],[0.7,"rgb(106, 137, 247)"],[1,"rgb(220, 220, 220)"]],
        autocolorscale = False,
        reversescale = True,
        marker = dict(
            line = dict (
                color = 'rgb(180,180,180)',
                width = 0.5
            ) ),
        colorbar = dict(
            title = 'Cases'),
      ) ]

layout = dict(
    title = TITLE,
    geo = dict(
        showframe = False,
        showcoastlines = False,
        projection = dict(
            type = 'Mercator'
        )
    )
)

print('\nMap using', len(only2015['country'].value_counts()), 'countries')

fig = dict( data=data, layout=layout )
py.iplot( fig, validate=False, filename='d3-world-map' )


Map using 192 countries


In [80]:
# FOR USE WITH ALL GRAPHS

def get_data_by_year_list(years, data, sort=None, ascending=True, first_x_only=None):

    result = []
    for year in years:
        
        curr_df = data[data['year'] == year]
        
        if sort:
            curr_df = curr_df.sort_values(sort, ascending=ascending)
            
        if first_x_only:
            curr_df = curr_df[:first_x_only]
        
        result.append(curr_df)
    
    return result

# Figure 2

In [81]:
tbcombined = get_data_by_year_list([2015, 2010, 2005, 2000], df_tb)

In [82]:
print('\nTotal sample size by year\n')

cols = ['population', 'tb_all_cases', 'hiv_cases', 'tb+hiv_cases', 'year']
kaka = pd.concat([tbcombined[0],tbcombined[1],tbcombined[2],tbcombined[3]], axis=0)[cols].groupby('year').sum()
kaka.columns = ['Total Population', 'TB Cases', 'HIV Cases', 'TB & HIV Cases']
kaka['TB & HIV Cases'] = kaka['TB & HIV Cases'].astype(np.int32)
kaka = kaka.reset_index()
kaka.columns = ['Year', 'Total Population', 'TB Cases', 'HIV Cases', 'TB & HIV Cases']
kaka


Total sample size by year



,Year,Total Population,TB Cases,HIV Cases,TB & HIV Cases
0,2000,3389504320,7915987,22911900,1092298
1,2005,3679784138,8468439,25009700,1282534
2,2010,3974117018,8388953,26081700,1234303
3,2015,4281947174,8207281,28280900,1011965


In [83]:
def create_x_vs_y_scatter(x, y, data, x_axis_title='X', y_axis_title='Y', title=None, no_outlier=False):

    # some initial required things
    
    colors = ['rgb(0, 255, 255)', 'rgb(250, 128, 114)', 'rgb(142, 69, 133)', 'rgb(218, 165, 32)']
    years = ['2015','2010','2005','2000']

    # for plotting
    traces = []
    buttons = []
    
    for i, curr_tb in enumerate(data):
              
        if no_outlier:
            mean = np.mean(curr_tb, axis=0)
            sd = np.std(curr_tb, axis=0)

            mean_x = mean.loc[x]
            mean_y = mean.loc[y]
            sd_x = sd.loc[x]
            sd_y = sd.loc[y]

            curr_tb = curr_tb[(curr_tb[x] > mean_x- 2*sd_x) & (curr_tb[y] > mean_y - 2*sd_y)][[x,y]]
            curr_tb = curr_tb[(curr_tb[x] < mean_x+ 2*sd_x) & (curr_tb[y] < mean_y + 2*sd_y)][[x,y]]
           
        
        slope, intercept, r_value, p_value, std_err = stats.linregress(curr_tb[x],curr_tb[y])
        line = slope*curr_tb[x]+intercept

        
        print('{} ({} countries):'.format(years[i], len(curr_tb)), end=' ')
        print(u'R\xb2', end=' ')
        print('= {0:.2f} |'.format(np.power(r_value,2)), end=' ')
        print('P-value = {}'.format(str(p_value)))
              

        
        trace1 = Scatter(
            x=curr_tb[x], y=curr_tb[y],
            mode='markers',
            name='Country',
            marker=dict(color=colors[i]),
            visible=False,
            showlegend=False,
        )

        trace2 = Scatter(
            x=curr_tb[x], 
            y=line, 
            mode='lines',
            showlegend=False,
            marker=dict(color=colors[i]),
            visible=False
        )

        # make 2015 visible by default
        if i == 0:
            trace1.visible=True
            trace2.visible=True
            
        # add buttons
        button_visibility = [False] * len(years) * 2
        button_visibility[2*i:2*i+2] = True, True
        button = dict( args=['visible', button_visibility], label=years[i], method='restyle' )
        
        buttons.append(button)
        traces += [trace1, trace2]

    
    if not title and no_outlier:
        title = '{} VS {} (outliers removed). Data source: WHO ({})'.format(x_axis_title, y_axis_title)
    elif not title:
        title = '{} VS {}. Data source: WHO'.format(x_axis_title, y_axis_title)
    else:
        title += '<br>Source: WHO'
        

    data = Data(traces)
    layout = Layout(
        title=title,
        yaxis=dict(title=y_axis_title),
        xaxis=dict(
            title=x_axis_title,
            showticklabels=True,   
            ticktext=[int(i/1000) for i in range(0,18001,2000)],
            tickvals=[i for i in range(0,18001,2000)]
        ),
        updatemenus=list([
            dict(
#                 x=-0.05,
#                 y=1200,
                yanchor='top',
                buttons=buttons,
            )
        ]),
        # showlegend=False,
    )
    fig = Figure(data=data, layout=layout)
    py.iplot(fig)
    
# create_x_vs_y_scatter(x='hiv_cases', y='tb_all_cases', data=tbcombined, x_axis_title='HIV Cases', y_axis_title='Tuberculosis Cases')

# Figure 3

In [84]:
###############################################################################
TITLE = 'HIV vs TB Prevalence'
XAXIS = 'HIV Cases (in thousands)'
YAXIS = 'Tuberculosis Cases'
###########################################################################

create_x_vs_y_scatter('hiv_cases_100k', 'tb_all_cases_100k', tbcombined, XAXIS, YAXIS, TITLE)

2015 (106 countries): R² = 0.44 | P-value = 8.3329877573e-15
2010 (105 countries): R² = 0.62 | P-value = 3.36874933079e-23
2005 (105 countries): R² = 0.66 | P-value = 1.14448184245e-25
2000 (105 countries): R² = 0.50 | P-value = 3.53707182784e-17


In [85]:
# GRAPH 2: HIV VS TB (removed outliers) (SCORE IS BAD. USE GRAPH 1 INSTEAD)

In [86]:
# create_x_vs_y_scatter(x='hiv_cases', y='tb_all_cases', data=tbcombined, x_axis_title='HIV Cases', y_axis_title='Tuberculosis Cases', no_outlier=True)

In [87]:
# create_x_vs_y_scatter('hiv_cases_100k', 'tb_all_cases_100k', tbcombined, 'HIV Cases (per 100k people)', 'Tuberculosis Cases (per 100k people)', no_outlier=True)

# Figure 4

In [88]:
def create_x_vs_multi_y_bar(x, y_list, data, data_decreasing_order, x_axis_title='X', y_axis_title='Y', legend_values=None, title=None, x_tick_label_current=None, x_tick_label_new=None):

    # some initial required things
    
    colors = ['rgb(153, 204, 255)', 'rgb(255, 153, 153)', 'rgb(153, 255, 153)']
    years = ['2015','2010','2005','2000']

    # for plotting
    traces = []
    buttons = []
    
    for i, curr_tb in enumerate(data): 
        
        for j, y in enumerate(y_list): 
        
            tempy = curr_tb[x]
            tempx = curr_tb[y]
            
            if data_decreasing_order:
                tempy = tempy[::-1]
                tempx = tempx[::-1]
            
            if legend_values:
                trace = go.Bar(
                    y=tempy,
                    x=tempx,
                    name=legend_values[j],
                    orientation='h',
                    visible=False,
                    marker=dict(
                        color=colors[j],
                        line=dict(
                            color='rgb(8,48,107)',
                            width=1.5,
                            )
                        ),
                    opacity=0.6
                )
            else:
                trace = go.Bar(
                    y=tempy,
                    x=tempx,
                    name=y,
                    orientation='h',
                    visible=False,
                    marker=dict(
                        color=colors[j],
                        line=dict(
                            color='rgb(8,48,107)',
                            width=1.5,
                            )
                        ),
                    opacity=0.6
                )
                

            # make 2015 visible by default
            if i == 0:
                trace.visible=True
                
            traces.append(trace)
            
        # add buttons 
        button_visibility = [False] * len(y_list) * len(years)
        button_visibility[len(y_list)*i:len(y_list)*i+len(y_list)] = [True] * len(y_list)
        button = dict( args=['visible', button_visibility], label=years[i], method='restyle' )
        
        buttons.append(button)

    
    if not title:
        title = '{} VS {}. Data source: WHO'.format(x_axis_title, y_axis_title)
    else:
        title += '<br>Source: WHO'
        
        
    if x_tick_label_new:
        xaxis = dict(
            title=x_axis_title,        
            showticklabels=True,   
            ticktext=x_tick_label_new,
            tickvals=x_tick_label_current
        )
    else:
        xaxis = dict(
            title=x_axis_title
        )
        
    data = Data(traces)
    layout = Layout(
        title=title,
        yaxis=dict(title=y_axis_title),
        xaxis=xaxis,
        updatemenus=list([
            dict(
                x=5.5,
                yanchor='bottom',
                buttons=buttons,
            )
        ]),
    )
    
    fig = Figure(data=data, layout=layout)
    py.iplot(fig)

# a.

In [106]:
df_tb.loc[df_tb['country']=='Bolivia (Plurinational State of)', 'country'] = 'Bolivia'

tbcombined_hivsort_top = get_data_by_year_list([2015, 2010, 2005, 2000], df_tb, 'hiv_cases_100k', ascending=False, first_x_only=10)
tbcombined_hivsort_bottom = get_data_by_year_list([2015, 2010, 2005, 2000], df_tb, 'hiv_cases_100k', ascending=True, first_x_only=40)
tbcombined_hivsort = []

tbcombined_hivsort_top_new = []
tbcombined_hivsort_bottom_new = []

for i in range(4):

    curr_top = tbcombined_hivsort_top[i]
    curr_bot = tbcombined_hivsort_bottom[i]
    
    # add space between top 5 and bottom 5
    curr_top.loc[99999] = ['', 0,0,0,0.0,0.0,0.0,0.0,0,0.0]
    
    
    # for the top 5 countries
    # 2015: swaziland, lesotho, south africa, namibia, zambia
    # 2005: swaziland, south africa, lesotho, namibia, zambia
    if i == 0 or i == 2:
        # for 2015
        condition_top = (curr_top['country'] == 'Swaziland') | (curr_top['country'] == 'Lesotho') | (curr_top['country'] == 'South Africa') | (curr_top['country'] == 'Namibia') | (curr_top['country'] == 'Zambia') | (curr_top['country'] == '')
    # 2010: swaziland, lesotho, south africa, namibia, zimbabwe
    elif i == 1:
         condition_top = (curr_top['country'] == 'Swaziland') | (curr_top['country'] == 'Lesotho') | (curr_top['country'] == 'South Africa') | (curr_top['country'] == 'Namibia') | (curr_top['country'] == 'Zimbabwe') | (curr_top['country'] == '')
    # 2000: botswana, swaziland, south africa, namibia, malawi
    elif i == 3:
        condition_top = (curr_top['country'] == 'Swaziland') | (curr_top['country'] == 'Botswana') | (curr_top['country'] == 'South Africa') | (curr_top['country'] == 'Namibia') | (curr_top['country'] == 'Malawi') | (curr_top['country'] == '')
    
    
    ########################################################################################################################
    
    # for the bottom 5 countries
    if i == 0:
        # 2015: niger, georgia, ecuador, nicaragua, mexico
        condition_bot = (curr_bot['country'] == 'Niger') | (curr_bot['country'] == 'Georgia') | (curr_bot['country'] == 'Ecuador') | (curr_bot['country'] == 'Nicaragua') | (curr_bot['country'] == 'Mexico') 
    
    elif i == 1:
        # 2010: nepal, plurinational state of, tajikistan, georgia, sudan
        condition_bot = (curr_bot['country'] == 'Bolivia') | (curr_bot['country'] == 'Tajikistan') | (curr_bot['country'] == 'Georgia') | (curr_bot['country'] == 'Sudan') | (curr_bot['country'] == 'Nepal')
        
    elif i == 2:
        # 2005: peru, nepal, state of, belarus, armenia/mexico
        condition_bot = (curr_bot['country'] == 'Bolivia') | (curr_bot['country'] == 'Peru') | (curr_bot['country'] == 'Nepal') | (curr_bot['country'] == 'Mexico') | (curr_bot['country'] == 'Belarus') 
        
    elif i == 3:
        # 2000: colombia, mexico, gatemala, argentina, chile
        condition_bot = (curr_bot['country'] == 'Colombia') | (curr_bot['country'] == 'Mexico') | (curr_bot['country'] == 'Guatemala') | (curr_bot['country'] == 'Argentina') | (curr_bot['country'] == 'Chile') 
    
    
    t=pd.concat([curr_bot[condition_bot], curr_top[condition_top][::-1]], axis=0)
    # change country name easier
    # t.loc[t['country']=='Bolivia (Plurinational State of)', 'country'] = 'Bolivia'
    tbcombined_hivsort.append(t)
    
    tbcombined_hivsort_bottom_new.append(curr_bot[condition_bot])
    tbcombined_hivsort_top_new.append(curr_top[condition_top])
    

In [107]:
years = [2015, 2010, 2005, 2000]
for i, c in enumerate(get_data_by_year_list(years, df_tb)):
    print('{}: {} countries'.format(years[i], len(c)))

###############################################################################
TITLE = 'Prevalence of HIV, TB, and HIV & TB in High and Low HIV Countries'
XAXIS = 'Cases (in thousands)'
YAXIS = 'Country'
LEGEND_VALUES = ['TB', 'HIV', 'TB & HIV']
X_TICK_LABEL_CURRENT = [i for i in range(0,16001,2000)]
X_TICK_LABEL_NEW = [int(i/1000) for i in range(0,16001,2000)]
###########################################################################


create_x_vs_multi_y_bar('country', ['tb_all_cases_100k', 'hiv_cases_100k', 'tb+hiv+cases_100k'], 
                        tbcombined_hivsort, False, x_axis_title=XAXIS, y_axis_title=YAXIS, legend_values=LEGEND_VALUES, title=TITLE, x_tick_label_current=X_TICK_LABEL_CURRENT, x_tick_label_new=X_TICK_LABEL_NEW)


2015: 106 countries
2010: 105 countries
2005: 105 countries
2000: 105 countries


# b.

In [109]:
###############################################################################
TITLE = 'Prevalence of HIV, TB, and HIV & TB in High HIV Countries'
XAXIS = 'Cases (in thousands)'
YAXIS = 'Country'
LEGEND_VALUES = ['TB', 'HIV', 'TB & HIV']
X_TICK_LABEL_CURRENT = [i for i in range(0,16001,2000)]
X_TICK_LABEL_NEW = [int(i/1000) for i in range(0,16001,2000)]
###########################################################################

for i in range(4):
    
    tbcombined_hivsort_top_new[i] = tbcombined_hivsort_top_new[i][:-1]

create_x_vs_multi_y_bar('country', ['tb_all_cases_100k', 'hiv_cases_100k', 'tb+hiv+cases_100k'], 
                        tbcombined_hivsort_top_new, True, x_axis_title=XAXIS, y_axis_title=YAXIS, legend_values=LEGEND_VALUES, title=TITLE, x_tick_label_current=X_TICK_LABEL_CURRENT, x_tick_label_new=X_TICK_LABEL_NEW)

# c.

In [110]:
###############################################################################
TITLE = 'Prevalence of HIV, TB, and HIV & TB in Low HIV Countries'
XAXIS = 'Cases'
YAXIS = 'Country'
LEGEND_VALUES = ['TB', 'HIV', 'TB & HIV']
###########################################################################

create_x_vs_multi_y_bar('country', ['tb_all_cases_100k', 'hiv_cases_100k', 'tb+hiv+cases_100k'], 
                        tbcombined_hivsort_bottom_new, False, x_axis_title=XAXIS, y_axis_title=YAXIS, legend_values=LEGEND_VALUES, title=TITLE)

# Figure 5

In [112]:
# read in extrapulmonary dataset 
columns = ['country', 'year', 'new_ep']
df_pul = pd.read_csv(get_object_storage_file_with_credentials_a4e60d264239489f8c0dbe5be7799f97('DefaultProjecteyeoffaustgmailcom', 'pulmonary.csv'))[columns]
df_pul.columns = ['country', 'year', 'extrapulm_tb_cases']

# hiv data by years
columns = ['year', 'population', 'country', 'hiv_cases_100k']
temp_df = df_tb[columns]

# combine the two
dfpul_combined = temp_df.merge(df_pul, on=['country','year'])

# calculate extra pulm cases per 100k
dfpul_combined['extrapulm_tb_cases_100k'] = np.round((dfpul_combined['extrapulm_tb_cases'] / dfpul_combined['population']) * 100000)

# get cases where extrapul and hiv are both not none
dfpul_combined = dfpul_combined[(dfpul_combined['hiv_cases_100k'].notnull()) & (dfpul_combined['extrapulm_tb_cases_100k'].notnull())]

In [113]:
###############################################################################
TITLE = 'HIV vs Extrapulmonary TB Prevalence'
XAXIS = 'HIV Cases (in thousands)'
YAXIS = 'Extrapulmonary TB Cases'
###########################################################################

scatter_data = get_data_by_year_list([2015, 2010, 2005, 2000], dfpul_combined)
create_x_vs_y_scatter('hiv_cases_100k', 'extrapulm_tb_cases_100k', scatter_data, XAXIS, YAXIS, TITLE)

2015 (105 countries): R² = 0.09 | P-value = 0.00197031967261
2010 (104 countries): R² = 0.26 | P-value = 2.73094495104e-08
2005 (101 countries): R² = 0.46 | P-value = 6.69799597789e-15
2000 (97 countries): R² = 0.25 | P-value = 1.97079359483e-07


# Figure 6

In [115]:
from sklearn.preprocessing import StandardScaler

In [120]:
dfpul_combined.loc[dfpul_combined['country']=='Iran (Islamic Republic of)', 'country'] = 'Iran'

pulmcombined_hivsort_top = get_data_by_year_list([2015, 2010, 2005, 2000], dfpul_combined, 'hiv_cases_100k', ascending=False)
pulmcombined_hivsort_bottom = get_data_by_year_list([2015, 2010, 2005, 2000], dfpul_combined, 'hiv_cases_100k', ascending=True)
pulmcombined_hivsort = []

temporary = []

for i in range(4):

    
    curr_top = pulmcombined_hivsort_top[i]
    curr_bot = pulmcombined_hivsort_bottom[i]
    
    s = StandardScaler()
    curr_top[['hiv_cases_100k', 'extrapulm_tb_cases_100k']] = s.fit_transform(curr_top[['hiv_cases_100k', 'extrapulm_tb_cases_100k']])
    curr_bot[['hiv_cases_100k', 'extrapulm_tb_cases_100k']] = s.fit_transform(curr_bot[['hiv_cases_100k', 'extrapulm_tb_cases_100k']])
    
    # add space between top 5 and bottom 5
    curr_top.loc[99999] = [0,0,'',0.0,0.0,0.0]

    
    # for the top 5 countries
    # 2015: swaziland, lesotho, south africa, namibia, zambia
    # 2005: swaziland, south africa, lesotho, namibia, zambia
    if i == 0 or i == 2:
        # for 2015
        condition_top = (curr_top['country'] == 'Swaziland') | (curr_top['country'] == 'Lesotho') | (curr_top['country'] == 'South Africa') | (curr_top['country'] == 'Namibia') | (curr_top['country'] == 'Zambia') | (curr_top['country'] == '')
    # 2010: swaziland, lesotho, south africa, namibia, zambia
    elif i == 1:
         condition_top = (curr_top['country'] == 'Swaziland') | (curr_top['country'] == 'Lesotho') | (curr_top['country'] == 'South Africa') | (curr_top['country'] == 'Namibia') | (curr_top['country'] == 'Zambia') | (curr_top['country'] == '')
    # 2000: # Botswana, Zimbabwe, Swaziland, Namibia, Malawi
    elif i == 3:
        condition_top = (curr_top['country'] == 'Botswana') | (curr_top['country'] == 'Zimbabwe') | (curr_top['country'] == 'Swaziland') | (curr_top['country'] == 'Namibia') | (curr_top['country'] == 'Malawi') | (curr_top['country'] == '')
    
    ########################################################################################################################
    
    # for the bottom 5 countries
    if i == 0:
        # 2015:  nicaragua, ecuador, niger, philippines, uzbekistan
        condition_bot = (curr_bot['country'] == 'Niger') | (curr_bot['country'] == 'Philippines') | (curr_bot['country'] == 'Ecuador') | (curr_bot['country'] == 'Nicaragua') | (curr_bot['country'] == 'Uzbekistan') 
    
    elif i == 1:
        # 2010: Egypt, Philippines, Lebanon, Iran (Islamic Republic of), Nicaragua
        condition_bot = (curr_bot['country'] == 'Egypt') | (curr_bot['country'] == 'Philippines') | (curr_bot['country'] == 'Lebanon') | (curr_bot['country'] == 'Iran') | (curr_bot['country'] == 'Nicaragua')
        
    elif i == 2:
        # 2005: Egypt, Lebanon, Cuba, Philippines, Kazakhstan
        condition_bot = (curr_bot['country'] == 'Egypt') | (curr_bot['country'] == 'Lebanon') | (curr_bot['country'] == 'Cuba') | (curr_bot['country'] == 'Philippines') | (curr_bot['country'] == 'Kazakhstan') 
        
    elif i == 3:
        # 2000: colombia, mexico, gatemala, argentina, chile
        condition_bot = (curr_bot['country'] == 'Colombia') | (curr_bot['country'] == 'Mexico') | (curr_bot['country'] == 'Guatemala') | (curr_bot['country'] == 'Argentina') | (curr_bot['country'] == 'Chile') 
    
    t=pd.concat([curr_bot[condition_bot], curr_top[condition_top][::-1]], axis=0)
    pulmcombined_hivsort.append(t)
    temporary.append(curr_bot[condition_bot])
    
    
years = [2015, 2010, 2005, 2000]
for i, c in enumerate(pulmcombined_hivsort_bottom):
    print('{}: {} countries'.format(years[i], len(c)))
    
    
###############################################################################
TITLE = 'Prevalence of HIV and Extrapulmonary TB in High and Low HIV Countries'
XAXIS = 'Standard deviation'
YAXIS = 'Country'
LEGEND_VALUES = ['HIV', 'Extrapulmonary TB']
###########################################################################

create_x_vs_multi_y_bar('country', ['hiv_cases_100k', 'extrapulm_tb_cases_100k'], 
                        pulmcombined_hivsort, False, x_axis_title=XAXIS, y_axis_title=YAXIS, legend_values=LEGEND_VALUES, title=TITLE)

2015: 105 countries
2010: 104 countries
2005: 101 countries
2000: 97 countries


In [123]:
columns = ['country', 'iso3', 'year', 'e_mort_tbhiv_num', 'e_mort_exc_tbhiv_num', 
          'e_inc_tbhiv_num', 'e_inc_num']

In [124]:
df = pd.read_csv(get_object_storage_file_with_credentials_a4e60d264239489f8c0dbe5be7799f97('DefaultProjecteyeoffaustgmailcom', 'basic_TB_data.csv'))[columns]

# rename columns for easier readability
df.columns = ['country', 'code', 'year', 'death_tb+hiv', 'death_tb_only',
              'cases_tb+hiv', 'cases_all_form']

# df.head()

In [125]:
# create new column consisting of people with tuberculosis and no hiv
# cases_total - cases_tb+hiv
df['cases_tb_only'] = df['cases_all_form'] - df['cases_tb+hiv']
df = df.drop('cases_all_form', axis=1)
# df.head()

In [126]:
# create percentages column

# mortality of those that had both tb and hiv
df['%_death_tb+hiv'] = df['death_tb+hiv'] / df['cases_tb+hiv']

# mortality of those that had only tb
df['%_death_tb_only'] = df['death_tb_only'] / df['cases_tb_only']

# drop non-% columns
col_to_drop = [
    'death_tb+hiv', 'cases_tb+hiv', 'death_tb_only', 'cases_tb_only'
]
df = df.drop(col_to_drop, axis=1)

# rounded % columns to 2 decimals and multiply by 100
df = df.round(2)
df[["%_death_tb+hiv","%_death_tb_only"]] *= 100

# remove null rows and change to integer data type
df = df[(df['%_death_tb+hiv'].notnull()) & (df['%_death_tb_only'].notnull())] 
df[["%_death_tb+hiv","%_death_tb_only"]] = df[["%_death_tb+hiv","%_death_tb_only"]].astype(np.int8)

# print(len(df))

# df.head()

In [127]:
# group into different years

df2015 = df[df['year'] == 2015]
df2010 = df[df['year'] == 2010]
df2005 = df[df['year'] == 2005]
df2000 = df[df['year'] == 2000]

# Figure 7

# a.

In [129]:
print("\nNumber of countries used for each year")
yaya = df[['year', 'country']].groupby('year').count()
yaya = yaya.reset_index()
yaya.columns = ['Year', 'Country']
yaya.transpose()


Number of countries used for each year


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
Year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
Country,157,157,158,163,168,174,173,172,174,180,185,183,184,183,185,181


# b.

In [269]:
#######################################################
TITLE = 'TB Mortality Rates (2000-2015)<br>Source: WHO'
########################################################


# Create and style traces
trace0 = go.Scatter(
    x = df_avg_deaths_by_year['year'],
    y = df_avg_deaths_by_year['%_death_tb+hiv'],
    name = 'With HIV',
    line = dict(
        color = ('rgb(205, 12, 24)'),
        width = 4)
)
trace1 = go.Scatter(
    x = df_avg_deaths_by_year['year'],
    y = df_avg_deaths_by_year['%_death_tb_only'],
    name = 'Without HIV',
    line = dict(
        color = ('rgb(22, 96, 167)'),
        width = 4,)
)
data = [trace0, trace1]

# Edit the layout
layout = dict(title = TITLE,
              xaxis = dict(title = 'Year'),
              yaxis = dict(title = 'Mortality Rate (%)'),
              )

fig = dict(data=data, layout=layout)
py.iplot(fig, filename='styled-line')

# c.

In [190]:
# across each year

df_avg_deaths_by_year = df.groupby('year').mean()
avg = df_avg_deaths_by_year['%_death_tb+hiv'] - df_avg_deaths_by_year['%_death_tb_only']

print("\nMortality rate difference in each year\n")
print(avg, '\n')

# across all years averaged total

t=df_avg_deaths_by_year['%_death_tb+hiv'] - df_avg_deaths_by_year['%_death_tb_only']
t=sum(t)/len(t)

print("Average mortality total\n")
print(t)


Mortality rate difference in each year

year
2000     6.980892
2001     8.019108
2002     8.930380
2003     7.828221
2004     6.910714
2005     7.655172
2006     8.479769
2007     8.482558
2008     8.316092
2009     8.511111
2010     9.243243
2011     9.601093
2012    10.032609
2013     9.054645
2014     8.848649
2015     9.154696
dtype: float64 

Average mortality total

8.50305948257


In [210]:
# Cochran-armitage test

In [213]:
coch = pd.DataFrame({
        'Scores': [i for i in range(1,17)],
        'Proportions': [0.589,0.603,0.618,0.608,0.594,0.604,0.618,0.612,0.614,0.620,0.632,0.644,0.653,0.638,0.634,0.639]
    })

# Figure 8

In [270]:
x = coch['Scores']
y = coch['Proportions']

# Create a trace
trace1 = go.Scatter(
    x = x,
    y = y,
    mode = 'markers',
    showlegend=False
)

slope, intercept, r_value, p_value, std_err = stats.linregress(x,y)
line = slope*x+intercept

trace2 = Scatter(
            x=x, 
            y=line, 
            mode='lines',
            showlegend=False
        )

annotation = go.Annotation(
                  x=3.5,
                  y=0.645,
                  text='R = {0:.4f}'.format(r_value),
                  showarrow=False,
                  font=go.Font(size=16)
                  )

layout = go.Layout(
                title='Cochran-Armitage Test for TB Mortality Rates',
                  annotations=[annotation]
                )

data = [trace1, trace2]
fig = go.Figure(data=data, layout=layout)

# Plot and embed in ipython notebook!
py.iplot(fig, filename='coch')

### Cochran-Armitage trend test (Asymptotic p-value) / Two-tailed test:		

|z| (Observed value)	7.568							
|z| (Critical value)	1.960							
p-value (Two-tailed)	< 0.0001							
alpha	0.05							
								
Test interpretation:								
H0: There is no association between the observed proportions and the score variable.								
Ha: There is an association between the observed proportions and the score variable.								
As the computed p-value is lower than the significance level alpha=0.05, one should reject the null hypothesis H0, and accept the alternative hypothesis Ha.								
The risk to reject the null hypothesis H0 while it is true is lower than 0.01%.				

### Cochran-Armitage trend test (Monte Carlo method - Number of simulations = 5000) / Two-tailed test:

|z| (Observed value)	7.568							
|z| (Critical value)	0.025							
p-value (Two-tailed)	< 0.0001							
alpha	0.05							
								
Test interpretation:								
H0: There is no association between the observed proportions and the score variable.								
Ha: There is an association between the observed proportions and the score variable.								
As the computed p-value is lower than the significance level alpha=0.05, one should reject the null hypothesis H0, and accept the alternative hypothesis Ha.									
The risk to reject the null hypothesis H0 while it is true is lower than 0.01%.					